# CPT Notebook - Continued Pre-Training

**Enhanced with W&B monitoring for real-time training visualization**

## Overview

This notebook performs Continued Pre-Training (CPT) on a selected LLM model for DFK detection.

### Key Features

- **Real-time W&B monitoring** - Watch training progress live in browser
- **Incremental data support** - Add new CSV, rerun, model continues learning
- **Flexible model selection** - Use any HuggingFace model (Qwen, Llama, SeaLLMs, etc.)
- **Automatic checkpoint detection** - Continues from existing LoRA if found

## Configuration Section

In [ ]:
# ============================================
# CONFIGURATION - EDIT THIS SECTION
# ============================================

# Base model (Unsloth pre-quantized 4-bit)
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"

# W&B Configuration (for real-time monitoring)
WANDB_PROJECT = "tim1-dfk"
WANDB_RUN_NAME = "cpt-run-"  # Timestamp will be added automatically
WANDB_ENTITY = None  # Set to your W&B username if not using default

# Training Configuration
NUM_EPOCHS = 3
BATCH_SIZE = 16  # Effective batch size
LEARNING_RATE = 2e-4

# Use existing CPT LoRA if found?
USE_EXISTING_LORA = True

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  Model: {MODEL_NAME}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  W&B Project: {WANDB_PROJECT}")
print(f"  Continue from existing LoRA: {USE_EXISTING_LORA}")
print("=" * 60)
print("")

# ============================================
# CONFIGURATION END
# ============================================

## Step 1: Setup Environment

In [ ]:
# Setup environment
import os
import sys
from pathlib import Path
from datetime import datetime

# Set HF cache to local disk (faster than Google Drive)
if 'COLAB_GPU' in os.environ:
    os.environ['HF_HOME'] = '/content/hf_cache'
    os.makedirs('/content/hf_cache', exist_ok=True)
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project directory
DRIVE_PATH = '/content/drive/MyDrive/Tim1-DFK'
if not os.path.exists(DRIVE_PATH):
    print('ERROR: Tim1-DFK folder not found on Google Drive!')
    print('Please upload your project folder to Google Drive first.')
    sys.exit(1)

os.chdir(DRIVE_PATH)
print(f'Working directory: {os.getcwd()}')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Step 2: Setup W&B (Weights & Biases)

**W&B provides real-time visualization of training progress.**

Benefits:
- Live loss curves in browser
- Compare multiple experiments
- Track hyperparameters
- Monitor GPU usage

In [ ]:
# Setup W&B for real-time training monitoring
print('\n' + '=' * 60)
print('SETTING UP W&B MONITORING')
print('=' * 60)

try:
    import wandb

    # Login to W&B (will prompt for API key if not already logged in)
    wandb.login()

    # Verify connection
    api = wandb.Api()
    viewer = api.viewer
    username = viewer.entity if hasattr(viewer, 'entity') else str(viewer)
    print(f'✅ W&B connected as: {username}')
    print(f"   Project: {WANDB_PROJECT}")

    # Set W&B API key as environment variable for scripts
    os.environ['WANDB_API_KEY'] = wandb.api.api_key
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY

    # Generate unique run name with timestamp
    timestamp = datetime.now().strftime("%m%d_%H%M%S")
    WANDB_RUN_FULL = f"{WANDB_RUN_NAME}{timestamp}"
    os.environ['WANDB_RUN_NAME'] = WANDB_RUN_FULL

    print(f"   Run name: {WANDB_RUN_FULL}")
    entity_display = WANDB_ENTITY or username
    print(f"   Monitor at: https://wandb.ai/{entity_display}/{WANDB_PROJECT}")

except ImportError:
    print('')
    print('⚠️  W&B not installed.')
    print('   Training will run without monitoring.')
    print('   To install, run: pip install wandb')
    os.environ['WANDB_DISABLED'] = 'true'

print('')
print('=' * 60)

## Step 3: Check Model Cache & Status

In [ ]:
# Check model cache and existing LoRA
print('\n' + '=' * 60)
print('CHECKING MODEL STATUS')
print('=' * 60)

# Check model cache (generic path from model name)
MODEL_CACHE = Path('/content/hf_cache/' + MODEL_NAME.split('/')[-1])

if MODEL_CACHE.exists():
    files = list(MODEL_CACHE.rglob('*'))
    total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1e9
    print(f'✅ Model cached locally')
    print(f'   Path: {MODEL_CACHE}')
    print(f'   Size: {total_size:.2f} GB')
    print(f'   Files: {len(files)}')
    MODEL_CACHED = True
else:
    print(f'ℹ️  Model not cached yet')
    print(f'   Will download from: {MODEL_NAME}')
    print(f'   Estimated download: ~5 GB (pre-quantized 4-bit)')
    print(f'   Will be cached for future sessions')
    MODEL_CACHED = False

print('')
# Check CPT LoRA
CPT_LORA_PATH = Path('outputs/cpt/lora_adapter')

if USE_EXISTING_LORA and CPT_LORA_PATH.exists():
    files = list(CPT_LORA_PATH.rglob('*'))
    total_size = sum(f.stat().st_size for f in files if f.is_file()) / (1024*1024)
    print(f'✅ Existing CPT LoRA found!')
    print(f'   Location: {CPT_LORA_PATH}')
    print(f'   Size: {total_size:.1f} MB')
    print(f'   Files: {len(files)}')
    print('')
    print('🔄 Continued Training Mode ACTIVE:')
    print('   ✓ Will load existing LoRA adapter')
    print('   ✓ Will train on top of it with new data')
    print('   ✓ Model knowledge accumulates across sessions')
    CONTINUE_FROM_CHECKPOINT = True
else:
    print(f'ℹ️  Starting fresh training (no existing LoRA)')
    CONTINUE_FROM_CHECKPOINT = False

## Step 4: Check CPT Data

In [ ]:
# Check CPT data files
print('\n' + '=' * 60)
print('CHECKING CPT DATA')
print('=' * 60)

CPT_DIR = Path('Dataset/CPT')
csv_files = list(CPT_DIR.glob('*.csv'))
processed_dir = CPT_DIR / 'processed'
corpus_file = processed_dir / 'cpt_corpus.txt'
stats_file = processed_dir / 'cpt_stats.json'

print(f'Available CSV files in Dataset/CPT/: {len(csv_files)}')
for f in csv_files:
    size_mb = f.stat().st_size / (1024*1024)
    print(f'  - {f.name} ({size_mb:.2f} MB)')

print('')
if corpus_file.exists():
    with open(corpus_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    size_kb = corpus_file.stat().st_size / 1024
    print(f'✅ Preprocessed corpus found!')
    print(f'   Documents: {len(lines):,}')
    print(f'   Size: {size_kb:.1f} KB')
    DATA_READY = True
else:
    print(f'❌ Preprocessed corpus not found')
    print(f'   Run preprocessing (Step 5) first')
    DATA_READY = False

if stats_file.exists():
    import json
    with open(stats_file) as f:
        stats = json.load(f)
    print('')
    print('Corpus Statistics:')
    print(f'  Total documents: {stats["total_documents"]:,}')
    print(f'  Total words: {stats["total_words"]:,}')
    print(f'  Avg words/doc: {stats["avg_words_per_doc"]:.1f}')
    print(f'  Duplicates removed: {stats["duplicates_removed"]:,}')
    print(f'  Source files: {len(stats["source_files"])}')

## Step 5: Install Dependencies

In [ ]:
%%capture
# Install Unsloth and dependencies (output captured to reduce noise)
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q hf_transfer
!pip install -q wandb
!pip install -q -r requirements.txt

## Step 6: Pre-Download Model

Download model Qwen3-8B terlebih dahulu agar **bisa di-resume** jika terputus.
Jika download terhenti, **jalankan cell ini lagi** — akan lanjut dari progress terakhir.

In [ ]:
# Pre-download model dengan resume support
# Jika download terhenti, jalankan cell ini lagi — akan lanjut dari progress terakhir

from huggingface_hub import snapshot_download
import shutil

MODEL_LOCAL = "/content/hf_cache/Qwen3-8B-bnb-4bit"
disk = shutil.disk_usage("/content")
free_gb = disk.free / 1e9
print(f"Disk tersedia: {free_gb:.1f} GB (butuh ~5 GB)")

if free_gb < 6:
    print("⚠️  Disk hampir penuh! Pertimbangkan hapus file yang tidak perlu.")

# snapshot_download otomatis resume jika sebelumnya terputus
print(f"\nDownloading {MODEL_NAME} ke {MODEL_LOCAL}...")
print("(Jika terputus, jalankan cell ini lagi untuk melanjutkan)\n")

snapshot_download(
    repo_id=MODEL_NAME,
    local_dir=MODEL_LOCAL,
    resume_download=True,
)

# Verifikasi
from pathlib import Path
files = list(Path(MODEL_LOCAL).rglob("*.safetensors"))
total_gb = sum(f.stat().st_size for f in files) / 1e9
print(f"\n✅ Model downloaded: {total_gb:.2f} GB ({len(files)} safetensor files)")
print(f"   Path: {MODEL_LOCAL}")

## Step 7: Preprocess CPT Data

Run this when you add new CSV files to `Dataset/CPT/`.
Cleans text, deduplicates, and outputs to `cpt_corpus.txt`.

In [ ]:
# Preprocess CPT data
# Run this when you add new CSV files to Dataset/CPT/

print('\n' + '=' * 60)
print('PREPROCESSING CPT DATA')
print('=' * 60)
print('This will:')
print('  1. Load all CSV files from Dataset/CPT/')
print('  2. Clean text (remove HTML, normalize whitespace)')
print('  3. Deduplicate (remove duplicate texts)')
print('  4. Save to Dataset/CPT/processed/cpt_corpus.txt')
print('')

!python scripts/preprocess_cpt.py

print('')
print('=' * 60)
print('PREPROCESSING COMPLETE')
print('=' * 60)
print('✅ Ready for training!')
print('')
print('Next step: Run training (Step 8)')

## Step 8: Configure & Run CPT Training

In [ ]:
# Run CPT training with W&B monitoring
print('\n' + '=' * 60)
print('STARTING CPT TRAINING')
print('=' * 60)
print('')
print('Configuration:')
print(f"  Base model: {MODEL_NAME}")
print(f'  LoRA rank: 16')
print(f'  Batch size (effective): {BATCH_SIZE}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Continue from checkpoint: {CONTINUE_FROM_CHECKPOINT}')
print('')
print('Monitoring:')
print(f'  W&B project: {WANDB_PROJECT}')
print('')
print('Notes:')
print('  - Output saved to: outputs/cpt/lora_adapter/')
print('  - If interrupted, run this cell again (continues from checkpoint)')
print('  - W&B tracks all metrics in real-time')
print('')
print('Estimated time: 1.5-3 hours (Tesla T4)')
print('=' * 60)
print('')

# Execute training
!python scripts/train_cpt.py --config configs/cpt_config.yaml

## Step 9: Real-Time Training Monitoring

Training is now running. **Keep this page open and periodically check the W&B dashboard.**

In [ ]:
# Check W&B run status
print('\n' + '=' * 60)
print('W&B RUN STATUS')
print('=' * 60)

try:
    import wandb
    
    if 'WANDB_DISABLED' not in os.environ:
        # Get current run
        api = wandb.Api()
        
        # Find recent run
        runs = api.runs(path=WANDB_PROJECT)
        
        if runs:
            latest_run = runs[0]
            print(f'✅ W&B Run Active!')
            print(f'   Run name: {latest_run.name}')
            print(f'   State: {latest_run.state}')
            print(f'   Created: {latest_run.created_at}')
            print('')
            print(f'📊 View dashboard: {latest_run.url}')
            print('')
            print('Metrics to monitor:')
            print('  - train/loss: Training loss over time')
            print('  - train/epoch: Current epoch')
            print('  - train/learning_rate: Learning rate schedule')
            print('  - train/grad_norm: Gradient norm (should be stable)')
            print('')
            print('What to look for:')
            print('  ✓ Loss decreasing (not increasing)')
            print('  ✓ Loss converging (flattening out)')
            print('  ✓ No sudden spikes (indicates data issues)')
        else:
            print('ℹ️  No W&B runs found yet.')
            print('   Check back after training starts.')
    
except Exception as e:
    print(f'ℹ️  Could not check W&B status: {e}')

## Step 10: Verify CPT Output

In [ ]:
# Verify CPT training output
import os
from pathlib import Path

print('\n' + '=' * 60)
print('CPT TRAINING VERIFICATION')
print('=' * 60)

output_dir = Path('outputs/cpt')
lora_path = output_dir / 'lora_adapter'

if lora_path.exists():
    print('✅ LoRA adapter saved successfully!')
    print(f'   Location: {lora_path}')
    print('')
    print('Files in LoRA adapter:')
    total_size = 0
    for f in sorted(lora_path.glob('*')):
        size_kb = f.stat().st_size / 1024
        total_size += size_kb
        print(f'  - {f.name} ({size_kb:.1f} KB)')
    
    print(f'   Total size: {total_size/1024:.1f} MB')
    print('')
    print('=' * 60)
    print('CPT TRAINING COMPLETE!')
    print('=' * 60)
    print('')
    print('Model Status:')
    print('  ✅ Base model: Trained with CPT data')
    print('  ✅ LoRA adapter: Ready for SFT')
    print('')
    print('Next steps:')
    print('  1. Open 2_SFT_Colab.ipynb for SFT training')
    print('  2. SFT will load this CPT LoRA automatically')
    print('')
    print('Future updates:')
    print('  - Add new CSV to Dataset/CPT/')
    print('  - Rerun preprocessing (Step 6)')
    print('  - Rerun training (Step 7)')
    print('  - Model will continue from this checkpoint')
    print('')
    print('W&B Monitoring:')
    print(f'  View full results: https://wandb.ai/{WANDB_ENTITY or "username"}/{WANDB_PROJECT}')
else:
    print('❌ LoRA adapter not found!')
    print('   Training may have failed or is incomplete.')
    print('')
    print('Troubleshooting:')
    print('  1. Check the training output above for errors')
    print('  2. Verify GPU is available (nvidia-smi)')
    print('  3. Check W&B dashboard for failure logs')

## Quick Reference: W&B Dashboard Guide

In [ ]:
# Print quick reference for W&B metrics interpretation
print('\n' + '=' * 60)
print('W&B METRICS INTERPRETATION GUIDE')
print('=' * 60)
print('')
print('📉 Training Loss (train/loss)')
print('   What it measures: How well model predicts next token')
print('   Good: Steadily decreasing, converges')
print('   Bad: Increasing, not decreasing')
print('')
print('📈 Learning Rate (train/learning_rate)')
print('   What it shows: Learning rate schedule over training')
print('   Good: Starts high, decreases per schedule')
print('')
print('📊 Gradient Norm (train/grad_norm)')
print('   What it measures: Size of gradient updates')
print('   Good: Stable, not spiking')
print('   Bad: Sudden spikes (check data/learning rate)')
print('')
print('⏱️  Epoch Time')
print('   What it shows: Time per epoch')
print('   Expected: 10-30 min/epoch depending on data size')
print('')
print('✅ Signs of Healthy Training:')
print('   ✓ Loss decreases smoothly')
print('   ✓ Loss plateaus (model learned what it can)')
print('   ✓ No sudden jumps')
print('   ✓ Gradient norm stable')
print('')
print('⚠️  Signs of Issues:')
print('   ⚠ Loss increasing')
print('   ⚠ Loss oscillating wildly')
print('   ⚠ Gradient norm spikes')
print('   ⚠ NaN or Inf values')
print('')
print('=' * 60)

---

## Summary

### What Just Happened

CPT (Continued Pre-Training) was performed on `{MODEL_NAME}` with real-time W&B monitoring.

In [ ]:
# Print training summary
print('CPT TRAINING SUMMARY')
print('=' * 60)
print('')
print('✓ Model loaded (or used cached version)')
print(f'  Model: {MODEL_NAME}')
print(f'  Cached: {MODEL_CACHED}')
print('')
print('✓ LoRA adapter applied')
print(f'  Rank: 16')
print(f'  Trainable params: ~35M out of 7B (0.5%)')
print('')
print('✓ Causal language modeling training')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Effective batch size: {BATCH_SIZE}')
print(f'  Learning rate: {LEARNING_RATE}')
print('')
print('✓ W&B monitoring active')
print(f'  Project: {WANDB_PROJECT}')
print(f'  Run: {WANDB_RUN_FULL}')
print('')
print('=' * 60)

### Output Generated
```
outputs/cpt/
└── lora_adapter/
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── tokenizer.json
```

### Time Taken
- Preprocessing: 1-5 minutes
- Training: 1.5-3.5 hours
- **Total: ~2-4 hours**

### Incremental Updates

**When you get new CPT data:**

1. Upload new CSV files to `Dataset/CPT/`
2. Rerun preprocessing (Step 6)
3. Rerun training (Step 7)

**What happens:**
- The script detects your existing LoRA adapter
- It loads the adapter and continues training
- New data is learned on top of existing knowledge
- W&B creates a new run for comparison
- No need to re-download or reinstall the model

**Result:** The model keeps learning and improving with each new dataset.

### Next Step

Proceed to **[2_SFT_Colab.ipynb](2_SFT_Colab.ipynb)** to perform Supervised Fine-Tuning on top of this CPT output.

### Changing Base Model

Want to use a different model? Edit the **Configuration Section** (Step 1):

```python
# Examples:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # Qwen model
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"  # Llama 3.1
MODEL_NAME = "mistralai/Mistral-7B-v0.3"  # Mistral
```

Then run Step 5-9 again. The model will be downloaded, cached, and trained with CPT.